In [1]:
from PIL import Image
from ultralytics import RTDETR
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import json
import glob
%matplotlib inline

In [2]:
# Check if CUDA (GPU) is available and set the device
if torch.cuda.is_available():
    device = torch.device("cuda:0") # Use the first GPU
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Training on CPU.")

Training on GPU: NVIDIA GeForce RTX 2060 SUPER


In [4]:
torch.cuda.empty_cache()

I made the following changes to the `Lib/site-packages/ultralytics/cfg/models/rt-detr/rtdetr-resnet50.yaml` file:

1.  **Updated the model description comment:**
    *   Changed `# Ultralytics RT-DETR-ResNet50 hybrid object detection model with P3/8 - P5/32 outputs` to `# Ultralytics RT-DETR-ResNet18 hybrid object detection model with P3/8 - P5/32 outputs`.

2.  **Modified the `backbone` section to reflect ResNet18 architecture:**
    *   The `ResNetLayer` arguments were adjusted to match a standard ResNet18 configuration, which typically uses 2 blocks per stage and basic blocks (block_type 2) instead of bottleneck blocks (block_type 3 or 4) used in ResNet50. The channel progression was also updated.
    *   Specifically, the lines:
        ```yaml
        - [-1, 1, ResNetLayer, [64, 64, 1, False, 3]] # 1
        - [-1, 1, ResNetLayer, [256, 128, 2, False, 4]] # 2
        - [-1, 1, ResNetLayer, [512, 256, 2, False, 6]] # 3
        - [-1, 1, ResNetLayer, [1024, 512, 2, False, 3]] # 4
        ```
        were changed to:
        ```yaml
        - [-1, 1, ResNetLayer, [64, 64, 2, False, 2]] # 1
        - [-1, 1, ResNetLayer, [128, 128, 2, False, 2]] # 2
        - [-1, 1, ResNetLayer, [256, 256, 2, False, 2]] # 3
        - [-1, 1, ResNetLayer, [512, 512, 2, False, 2]] # 4
        ```

3.  **Adjusted the `head` section to match ResNet18 output channels:**
    *   The input channels for the `AIFI` module were changed from `1024` to `512`, as ResNet18 typically outputs 512 channels from its final stage.
    *   Specifically, the line:
        ```yaml
        - [-1, 1, AIFI, [1024, 8]]
        ```
        was changed to:
        ```yaml
        - [-1, 1, AIFI, [512, 8]]
        ```

In [4]:
model = RTDETR()

In [5]:
result = model.train(
    data='Shrimp-larvae-detection-1/data.yaml',
    epochs = 30,
    batch = 8,
    device=0
)

Ultralytics 8.3.204  Python-3.13.7 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Shrimp-larvae-detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

e:\PD1ModelTrainings\DLenv\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      10.4G     0.6988     0.4552      0.118         99        640: 100% ━━━━━━━━━━━━ 518/518 0.5it/s 15:53<1.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 2.7it/s 9.2s0.3s
                   all        386      13790      0.831      0.878      0.866      0.454

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/30      7.29G     0.6484     0.4368    0.06821        382        640: 0% ──────────── 0/518  1.1s

e:\PD1ModelTrainings\DLenv\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      8.22G     0.5388      0.443    0.06334        153        640: 100% ━━━━━━━━━━━━ 518/518 1.0it/s 8:34<0.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 2.9it/s 8.5s0.3s
                   all        386      13790      0.871      0.908      0.926      0.571

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

      28/30      8.76G     0.3577     0.4058    0.03924         60        640: 100% ━━━━━━━━━━━━ 518/518 0.9it/s 10:06<1.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 3.4it/s 7.3s0.3s
                   all        386      13790      0.889      0.929      0.956      0.655

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      29/30      6.54G     0.3298     0.4049    0.03206        214        640: 0% ──────────── 0/518  0.9s

e:\PD1ModelTrainings\DLenv\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      8.99G      0.342     0.4009     0.0376         64        640: 100% ━━━━━━━━━━━━ 518/518 1.2it/s 7:01<0.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 3.4it/s 7.3s0.3s
                   all        386      13790       0.89      0.929      0.954      0.648

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/30      7.28G     0.3707     0.3932    0.04323        306        640: 0% ──────────── 0/518  0.7s

e:\PD1ModelTrainings\DLenv\Lib\site-packages\torch\autograd\graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:97.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      9.56G     0.3346     0.3984    0.03646         33        640: 100% ━━━━━━━━━━━━ 518/518 1.2it/s 7:01<0.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 3.4it/s 7.3s0.3s
                   all        386      13790      0.888      0.927      0.956      0.649

30 epochs completed in 6.777 hours.
Optimizer stripped from E:\PD1ModelTrainings\PD1ModelTrainingCodes\runs\detect\train6\weights\last.pt, 66.1MB
Optimizer stripped from E:\PD1ModelTrainings\PD1ModelTrainingCodes\runs\detect\train6\weights\best.pt, 66.1MB

Validating E:\PD1ModelTrainings\PD1ModelTrainingCodes\runs\detect\train6\weights\best.pt...
Ultralytics 8.3.204  Python-3.13.7 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)
rt-detr-l summary: 302 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 3.1it/s 8.0s

In [4]:
model_test = RTDETR('runs/detect/RTDETRFinalweights/best.pt')

In [5]:
import torch
import time

# ✅ Clear GPU cache before test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# ✅ Warm-up GPU (optional for consistent timing)
if device.type == "cuda":
    dummy = torch.randn(1, 3, 640, 640).to(device)
    _ = model_test(dummy)
    torch.cuda.synchronize()

# ✅ Measure inference time + GPU memory
if device.type == "cuda":
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    start_event.record()
else:
    start_time = time.time()

# --- INFERENCE ---
test_pred = model_test.predict(
    source="test.jpg",
    show_labels=False,
    show_conf=False,
    save=True
)

# ✅ End timing
if device.type == "cuda":
    end_event.record()
    torch.cuda.synchronize()
    inference_time_ms = start_event.elapsed_time(end_event)
    max_memory = torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # MB
else:
    inference_time_ms = (time.time() - start_time) * 1000
    max_memory = 0.0

# ✅ Print metrics
print(f"⚙️ Inference Time: {inference_time_ms:.2f} ms")
print(f"💾 GPU Memory Usage: {max_memory:.2f} MB")



WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.099058628082275. Dividing input by 255.
0: 640x640 (no detections), 85.4ms
Speed: 0.0ms preprocess, 85.4ms inference, 4.5ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 e:\PD1ModelTrainings\PD1ModelTrainingCodes\test.jpg: 640x640 268 s, 48.3ms
Speed: 5.5ms preprocess, 48.3ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)
Results saved to E:\PD1ModelTrainings\PD1ModelTrainingCodes\runs\detect\predict4
⚙️ Inference Time: 136.05 ms
💾 GPU Memory Usage: 240.53 MB
